# Demo 08 — External models via MaaS

OpenAI-compatible **`POST …/v1/chat/completions`** for llm-katan **`ExternalModel`** sims (`sim-chat`, optional `sim-chat-2` / `sim-messages`). Prefer **body-based routing**; path-based preset included for contrast.

**Prereq:** Demo 08 applied (`manifests/external-model.yaml`); IPP running; cluster can reach `3-132-132-211.sslip.io`. Python 3.9+ stdlib.

**Credentials:** Paste **`DEMO_API_KEY`** (minted against **`demo08-hybrid-catalog`**) or set `MAAS_API_KEY` / `API_KEY`.


## Demo quick swap

| Variable | Effect |
|----------|--------|
| `DEMO_MAAS_BASE` | Gateway origin override. |
| `DEMO_API_KEY` | MaaS `sk-oai-*` override. |
| `USE_BBR` | `True` (default) → `{MAAS_BASE}/v1/chat/completions`; `False` → path-based `/llm/sim-chat/…`. |
| `EXTERNAL_MODEL_ID` | Model id in the chat body (default `sim-chat`). |


In [ ]:
DEMO_MAAS_BASE = ""
DEMO_API_KEY = ""
USE_BBR = True
EXTERNAL_MODEL_ID = "sim-chat"
USER_MESSAGE = "What model am I using? Answer in one short sentence."
MAX_TOKENS = 128


## Load config + resolve URL


In [ ]:
import json
import os
import ssl
import urllib.error
import urllib.request

_mb = globals().get("DEMO_MAAS_BASE", "")
if isinstance(_mb, str) and _mb.strip():
    MAAS_BASE = _mb.strip().rstrip("/")
else:
    MAAS_BASE = os.environ.get("MAAS_BASE", "https://maas.YOUR_DOMAIN_HERE").strip().rstrip("/")

_ak = globals().get("DEMO_API_KEY", "")
if isinstance(_ak, str) and _ak.strip():
    API_KEY = _ak.strip()
else:
    API_KEY = (os.environ.get("MAAS_API_KEY") or os.environ.get("API_KEY") or "").strip()

VERIFY_TLS = os.environ.get("VERIFY_TLS", "").lower() in ("1", "true", "yes")
USE_BBR = bool(globals().get("USE_BBR", True))
EXTERNAL_MODEL_ID = (globals().get("EXTERNAL_MODEL_ID") or "sim-chat").strip()
USER_MESSAGE = globals().get("USER_MESSAGE") or "What model am I using?"
MAX_TOKENS = int(globals().get("MAX_TOKENS") or 128)

if not API_KEY:
    raise SystemExit("Set DEMO_API_KEY or MAAS_API_KEY / API_KEY.")

MODELS_URL = f"{MAAS_BASE}/maas-api/v1/models"


def _ssl_ctx():
    ctx = ssl.create_default_context()
    if not VERIFY_TLS:
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
    return ctx


def http_json(method, url, *, token=None, data=None, timeout=300):
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    body = json.dumps(data).encode("utf-8") if data is not None else None
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=_ssl_ctx(), timeout=timeout) as resp:
            raw = resp.read().decode("utf-8")
            return resp.status, json.loads(raw) if raw else {}
    except urllib.error.HTTPError as e:
        err = e.read().decode("utf-8", errors="replace")
        try:
            parsed = json.loads(err) if err else {}
        except json.JSONDecodeError:
            parsed = {"_raw": err}
        raise RuntimeError(f"HTTP {e.code}: {parsed}") from None


# Resolve path-based URL from discovery when available
PATH_CHAT_URL = f"{MAAS_BASE}/llm/{EXTERNAL_MODEL_ID}/v1/chat/completions"
DISCOVERED_ID = EXTERNAL_MODEL_ID
try:
    _, models_body = http_json("GET", MODELS_URL, token=API_KEY)
    for m in models_body.get("data") or []:
        mid = m.get("id") or m.get("name") or ""
        if "sim-chat" in mid.lower() or mid == EXTERNAL_MODEL_ID:
            DISCOVERED_ID = mid
            if m.get("url"):
                PATH_CHAT_URL = m["url"].rstrip("/") + "/v1/chat/completions"
            break
except RuntimeError as e:
    print("Discovery warning:", e)

BBR_CHAT_URL = f"{MAAS_BASE}/v1/chat/completions"
CHAT_URL = BBR_CHAT_URL if USE_BBR else PATH_CHAT_URL

print("USE_BBR       :", USE_BBR)
print("model id      :", DISCOVERED_ID)
print("POST URL      :", CHAT_URL)
print("VERIFY_TLS    :", VERIFY_TLS)
print("API key set   :", bool(API_KEY))


## Chat completion (external)

Non-streaming JSON. Parses **`choices[0].message.content`**. User MaaS key stays at the gateway; IPP injects the provider Secret toward OpenAI.


In [ ]:
def _assistant_text(obj: dict) -> str:
    choices = obj.get("choices") or []
    if not choices:
        return ""
    msg = (choices[0].get("message") or {}) if isinstance(choices[0], dict) else {}
    c = msg.get("content")
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for b in c:
            if isinstance(b, dict):
                t = b.get("text") or b.get("content")
                if isinstance(t, str):
                    parts.append(t)
            elif isinstance(b, str):
                parts.append(b)
        return "".join(parts)
    return ""


payload = {
    "model": DISCOVERED_ID,
    "messages": [{"role": "user", "content": USER_MESSAGE}],
    "max_tokens": MAX_TOKENS,
}

try:
    status, obj = http_json("POST", CHAT_URL, token=API_KEY, data=payload)
except RuntimeError as e:
    print(e)
    print("Tip: oc get externalmodel,maasmodelref -n llm | grep sim-; check IPP can reach 3-132-132-211.sslip.io.")
    raise

text = _assistant_text(obj) if isinstance(obj, dict) else ""
print(text if text else "(no content)", "\n")
if isinstance(obj, dict) and obj.get("usage"):
    print("usage:", obj["usage"])
print("HTTP", status)


## Optional — flip to path-based

Set **`USE_BBR = False`** in quick swap and re-run **Load config** + **Chat completion**, or run the cell below without reconfiguring.


In [ ]:
print("Path-based POST:", PATH_CHAT_URL)
try:
    status, obj = http_json(
        "POST",
        PATH_CHAT_URL,
        token=API_KEY,
        data={
            "model": DISCOVERED_ID,
            "messages": [{"role": "user", "content": USER_MESSAGE}],
            "max_tokens": MAX_TOKENS,
        },
    )
    print(_assistant_text(obj) if isinstance(obj, dict) else "")
    print("HTTP", status)
except RuntimeError as e:
    print(e)
